In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN attached:", True)
except Exception as e:
    print("HF_TOKEN missing (benchmarks still run; Hub push of results will skip):", e)


In [ ]:
!rm -rf /kaggle/working/arc && git clone --branch stage-a-cpt https://github.com/Nyvo2010/arc.git /kaggle/working/arc
!pip install -q -r /kaggle/working/arc/requirements-kaggle.txt huggingface_hub


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="jetmoe/jetmoe-8b", local_dir="/kaggle/working/jetmoe-8b")
print("weights ready")


In [ ]:
import glob, json, os
from pathlib import Path
adapter_dirs = {}
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    hits = sorted(glob.glob("/kaggle/input/*/checkpoints/phase1-tier-b/*/{variant}/adapters-best"))
    if hits:
        adapter_dirs[variant] = hits[0]
        print(f"adapter [{variant}] ->{hits[0]}")
    else:
        print(f"!! NO adapter output for [{variant}]")
if not adapter_dirs:
    print("INPUT DIRS:", os.listdir("/kaggle/input"))
    raise SystemExit("No Tier B checkpoint outputs found - were the training kernels completed and attached?")
open("/kaggle/working/adapters.json", "w").write(json.dumps(adapter_dirs, indent=2))
for variant, d in sorted(adapter_dirs.items()):
    rd = Path(d).resolve().parent
    best = json.load(open(rd / "best.json")) if (rd / "best.json").exists() else {}
    st = json.load(open(rd / "train_state.json")) if (rd / "train_state.json").exists() else {}
    print(variant, "| best.best_val_loss:", best.get("best_val_loss"),
          "| tokens:", st.get("tokens_processed"), "| step:", st.get("step"))


In [ ]:
import json, os, sys, torch
# SMOKE: reproducibility of the Hub-published repos. Load ONE variant through
# AutoModelForCausalLM.from_pretrained(repo, trust_remote_code=True) and verify
# its logits match the local JIT build with the same base + adapter weights.
adapter_dirs = json.load(open("/kaggle/working/adapters.json"))
variant = "block_adaptive"

# --- local reference build (training-time code, same inputs) ---
sys.path.insert(0, "/kaggle/working/arc/src")
from arc.recurrence.builder import build_model
from arc.models.registry import create_adapter
from peft import PeftModel

adap = create_adapter("/kaggle/working/jetmoe-8b", device_map="auto")
adap.hf_model = PeftModel.from_pretrained(adap.hf_model, adapter_dirs[variant])
adap.net = adap.hf_model.model
adap.head = adap.hf_model.lm_head
adap.hf_model.eval()
local = build_model("block", adap, max_loops=4).eval()

repo = "Nyvo/arc-jetmoe-block-adaptive"
from transformers import AutoModelForCausalLM
hub = AutoModelForCausalLM.from_pretrained(repo, trust_remote_code=True,
                                           token=os.environ.get("HF_TOKEN") or True)
hub.eval()

ids = torch.tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]])
with torch.no_grad():
    ids = ids.to("cuda" if torch.cuda.is_available() else "cpu")
    l_out = local(ids)
    h_out = hub(ids)
l_logits = l_out.logits.float()
h_logits = h_out.logits.float()
diff = (l_logits - h_logits).abs().max().item()
print(f"[smoke] {variant} | local-hub logits max-abs-diff = {diff:.2e}")
assert diff < 1e-3, f"Hub repo and local build disagree (diff={diff})"
print("[smoke] OK: Hub repo reproduces the trained behavior")


In [ ]:
import json, os
adapter_dirs = json.load(open("/kaggle/working/adapters.json"))
adap = ",".join(f"{k}={v}" for k, v in sorted(adapter_dirs.items()))
print(adap)
# PRIMARY RUN: real Policy-T halt head (budgeted) for base + all 3 variants.
!cd /kaggle/working/arc && python scripts/run_benchmarks.py \
  --base /kaggle/working/jetmoe-8b \
  --model base,model_adaptive,block_adaptive,layer_adaptive \
  --adapters {adap} \
  --budgeted \
  --max_loops 4 \
  --limits wikitext=1000 \
  --out /kaggle/working/benchmarks-tierb-budgeted.csv


In [ ]:
import json, subprocess, sys
adapter_dirs = json.load(open("/kaggle/working/adapters.json"))
adap = ",".join(f"{k}={v}" for k, v in sorted(adapter_dirs.items()))
# RECURRENCE CAP SENSITIVITY: budgeted eval at tighter caps (compute frontier).
for ml in (2, 3):
    out = f"/kaggle/working/benchmarks-tierb-loops{ml}.csv"
    cmd = [sys.executable, "scripts/run_benchmarks.py",
           "--base", "/kaggle/working/jetmoe-8b",
           "--model", "model_adaptive,block_adaptive,layer_adaptive",
           "--adapters", adap,
           "--budgeted",
           "--max_loops", str(ml),
           "--tasks", "arc_easy,arc_challenge,hellaswag,piqa,winogrande,boolq,sciq",
           "--limits", "arc_easy=150,arc_challenge=150,hellaswag=150,piqa=150,winogrande=150,boolq=150,sciq=150",
           "--out", out]
    print(">>> max_loops", ml)
    subprocess.run(cmd, cwd="/kaggle/working/arc", check=True)


In [ ]:
import csv, glob, math, os, shutil
from pathlib import Path
os.makedirs("/kaggle/output/benchmarks", exist_ok=True)
csvs = sorted(glob.glob("/kaggle/working/benchmarks-*.csv"))
for c in csvs:
    shutil.copy(c, f"/kaggle/output/benchmarks/{{Path(c).name}}")
    print("saved", c)
rows = []
for c in csvs:
    rows.extend(csv.DictReader(open(c)))
mcq = ["arc_easy", "arc_challenge", "hellaswag", "piqa", "winogrande", "boolq", "sciq"]
def g(key, task, budgeted, loops):
    for r in rows:
        if (r["model"] == key and r["task"] == task
                and str(r.get("budgeted", "False")) == str(budgeted)
                and int(r.get("max_loops", 4) or 4) == int(loops or 4)):
            return r
    return None
for key in ["base", "model_adaptive", "block_adaptive", "layer_adaptive"]:
    print("\n===", key, "===")
    for task in mcq:
        r = g(key, task, True, 4)
        if r:
            print(f"  {{task:14s}} acc={{float(r['acc'])*100:5.1f}}  acc_norm={{float(r['acc_norm'])*100:5.1f}}  "
                  f"loops={{r.get('avg_loops_per_item','-')}}  tok/it={{r.get('avg_tokens_per_item','-')}}  "
                  f"tok/s={{r.get('tokens_per_s','-')}}  GFLOP/s={{r.get('flops_per_s','-')}}")
            try:
                print(f"       flops/it={{float(r.get('avg_flops_per_item',0))/1e12:6.2f}}T  "
                      f"elapsed={{r.get('elapsed_s','-')}}s  items/s={{r.get('items_per_s','-')}}")
            except (TypeError, ValueError):
                pass
    r = g(key, "wikitext", True, 4)
    if r: print(f"  wikitext      ppl={{float(r['ppl']):6.1f}}  loops={{r.get('avg_loops_per_item','-')}}  "
                f"tok/s={{r.get('tokens_per_s','-')}}  GFLOP/s={{r.get('flops_per_s','-')}}")


In [ ]:
import json, os
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi
    api = HfApi()
    repo = "Nyvo/arc-jetmoe-recurrence-phase1"
    for f in sorted(glob.glob("/kaggle/output/benchmarks/*.csv")):
        api.upload_file(path_or_fileobj=f, path_in_repo=f"benches/{{os.path.basename(f)}}",
                        repo_id=repo, token=os.environ["HF_TOKEN"])
        print("pushed", f)
else:
    print("HF_TOKEN absent: results kept in Kaggle output; push manually")
